# ZuCo sentiment from classical EEG features

Classical stats of the raw EEG plus ZuCo's precomputed band-power means, then a
linear classifier. A plain CPU runtime is fine here, no GPU needed.
Mount Drive (the `.mat` files are large and live there), point `MAT_DIR` at them
and `RESULTS_DIR` at where you want the output, then run top to bottom.

In [ ]:
# clone (or pull) the code, install deps
import os
REPO = 'zuco-eeg-classical-features'
if not os.path.exists(REPO):
    !git clone https://github.com/parmisbathaeiyan/zuco-eeg-classical-features.git $REPO
else:
    !cd $REPO && git pull --ff-only
%cd $REPO
!pip install -q -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# folder with the EEG results*_SR.mat files
MAT_DIR = '/content/drive/MyDrive/Thesis/Data/zuco_og_raw'

# where to save results (on Drive so they survive a runtime reset). Created if
# it doesn't exist; change to e.g. 'results' to keep them on the Colab disk.
RESULTS_DIR = '/content/drive/MyDrive/Thesis/Results/zuco_eeg_classical'

In [ ]:
# 1. extract features -> features/<SUBJ>.npz  (slow; resumable, skips cached)
!python extract_features.py --mat-dir "$MAT_DIR" --out-dir features

In [ ]:
# 2. classify: subject-specific + pooled, results written under RESULTS_DIR
!python run.py --features-dir features --output-dir "$RESULTS_DIR" --classifier logreg

In [ ]:
# 3. tables -> RESULTS_DIR/tables/{summary.csv,summary.md,per_subject_*.csv}
!python make_tables.py --results-dir "$RESULTS_DIR"

from IPython.display import Markdown, display
display(Markdown(open(os.path.join(RESULTS_DIR, 'tables', 'summary.md')).read()))

In [ ]:
# the plots run.py saved
from IPython.display import Image, display
for name in ['plots/subject_accuracy_logreg.png',
             'plots/cm_pooled_logreg.png']:
    p = os.path.join(RESULTS_DIR, name)
    if os.path.exists(p):
        display(Image(p))